# Triton Kernel Starter

Matrix multiplication scaffold using Triton, replacing the initial vector-add example. Includes environment checks, a tiled matmul kernel, and notes on how to adapt it toward the paper's sparse formulation.



In [ ]:
import torch

# Import Triton and its DSL for GPU kernels
try:
    import triton
    import triton.language as tl
except ImportError as exc:
    raise ImportError(
        "Install dependencies first: pip install -r requirements.txt"
    ) from exc

# Quick environment report for reproducibility/debugging
print(f"Torch version: {torch.__version__}")
print(f"Triton version: {triton.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Kernels only launch on CUDA; CPU path is for editing/tests only
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")



## Sample inputs

Create simple input tensors for dense matrix multiplication (C = A @ B). Adjust shapes/dtypes to mirror the paper's kernel assumptions.



In [ ]:
torch.manual_seed(0)

# Matrix dimensions: (M x K) @ (K x N) -> (M x N)
# Keep sizes modest so the kernel runs quickly; tune to match the paper later.
M, N, K = 512, 512, 512

# Use float16 on GPU for speed; float32 fallback on CPU for correctness checks
dtype = torch.float16 if DEVICE == "cuda" else torch.float32
A = torch.randn((M, K), device=DEVICE, dtype=dtype)
B = torch.randn((K, N), device=DEVICE, dtype=dtype)



## Triton kernel: matrix multiplication

Tiled matmul kernel using Triton. Replace the body to match the paper's sparse kernel once ready.



In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    *, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr = 8,
):
    """
    Compute C = A @ B for a single block of C.

    Args are raw pointers/strides so the kernel works with non-contiguous inputs.
    BLOCK sizes and GROUP_M can be tuned to mirror the paper's tile sizes.
    """
    # Program IDs identify which block of C this instance is responsible for
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    # Reorder program ids to promote L2 locality along M dimension
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    pid = pid_n * num_pid_m + pid_m
    group_id = pid // GROUP_M
    first_pid_m = group_id * GROUP_M
    pid_m = first_pid_m + (pid % GROUP_M)
    pid_n = pid // num_pid_m

    # Offsets for this block
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    # Initialize accumulator in FP32 for better precision
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # Loop over K dimension in BLOCK_K chunks
    for k0 in range(0, K, BLOCK_K):
        offs_k = k0 + tl.arange(0, BLOCK_K)

        a = tl.load(
            a_ptr + (offs_m[:, None] * stride_am) + (offs_k[None, :] * stride_ak),
            mask=(offs_m[:, None] < M) & (offs_k[None, :] < K),
            other=0.0,
        )
        b = tl.load(
            b_ptr + (offs_k[:, None] * stride_bk) + (offs_n[None, :] * stride_bn),
            mask=(offs_k[:, None] < K) & (offs_n[None, :] < N),
            other=0.0,
        )

        # Fused matmul for this tile
        acc += tl.dot(a, b)

    # Write back to C with boundary check
    c = acc.to(tl.float16 if tl.float16 in (a.dtype, b.dtype) else tl.float32)
    tl.store(
        c_ptr + (offs_m[:, None] * stride_cm) + (offs_n[None, :] * stride_cn),
        c,
        mask=(offs_m[:, None] < M) & (offs_n[None, :] < N),
    )


def triton_matmul(A: torch.Tensor, B: torch.Tensor, *, block_m=128, block_n=128, block_k=32, num_warps=4):
    """Launch Triton matmul. Input shapes: (M, K) @ (K, N)."""
    if A.device.type != "cuda" or B.device.type != "cuda":
        raise RuntimeError("Move tensors to CUDA before launching Triton kernels")
    assert A.shape[1] == B.shape[0], "Incompatible shapes for matmul"

    M, K = A.shape
    K_, N = B.shape
    assert K == K_, "Inner dimensions must match"

    # Allocate output in FP16/FP32 following A's dtype
    C = torch.empty((M, N), device=A.device, dtype=A.dtype)

    # Grid is 2D: one block per tile of C
    grid = (
        triton.cdiv(M, block_m),
        triton.cdiv(N, block_n),
    )

    matmul_kernel[grid](
        A, B, C,
        M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
        GROUP_M=8,
        num_warps=num_warps,
        num_stages=2,
    )
    return C



In [ ]:
if torch.cuda.is_available():
    # Move inputs to CUDA for Triton launch
    A_cuda = A.to("cuda") if A.device.type != "cuda" else A
    B_cuda = B.to("cuda") if B.device.type != "cuda" else B

    # Run Triton matmul and compare against torch.matmul for correctness
    C_triton = triton_matmul(A_cuda, B_cuda)
    C_ref = torch.matmul(A_cuda, B_cuda)
    torch.testing.assert_close(C_triton, C_ref, rtol=1e-2, atol=1e-2)
    print("Matmul passed on CUDA device")
else:
    print(
        "CUDA not available; skipping Triton kernel launch. Edit the kernel and push to a GPU-enabled machine to validate."
    )



## Next steps
- Adapt the matmul body to implement the paper's sparse kernel (data layout, masking, and accumulation rules).
- Tune tile sizes (`BLOCK_M/N/K`, `num_warps`, `num_stages`, `GROUP_M`) to match the CUDA hierarchy and the target GPU.
- Add quick perf checks (GFLOPs, occupancy estimates) and log paper baselines for comparison.

